# 09 — ML Bridge-Type Selection — Explainability & Robustness

## Objective

Analyse which of the four agreed project inputs influence:
1. `Bauwerksart` classification
2. `zustandsnote` estimation

The analysis is diagnostic only. It does not modify the frozen production model and does not create a structural design.


In [1]:
from getpass import getpass
from pathlib import Path
import os
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sqlalchemy import create_engine, URL, text
from pyproj import Transformer

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import ExtraTreesClassifier, ExtraTreesRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import GroupShuffleSplit

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

def find_project_root():
    env_root = os.getenv("BRIDGE_PROJECT_ROOT")
    if env_root:
        root = Path(env_root).expanduser().resolve()
        if (root / "Dataset_PlanA-B").exists():
            return root
        raise FileNotFoundError(f"BRIDGE_PROJECT_ROOT does not contain Dataset_PlanA-B: {root}")
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "Dataset_PlanA-B").exists():
            return candidate
    raise FileNotFoundError("Project root not found. Set BRIDGE_PROJECT_ROOT to the project folder.")

PROJECT_ROOT = find_project_root()
DATASET_ROOT = PROJECT_ROOT / "Dataset_PlanA-B"
OUTPUT_ROOT = PROJECT_ROOT / "Output_PlanA-B"
OUTPUT_DIR = OUTPUT_ROOT / "09_ML_Bridge_Type_Selection_Explainability_Robustness"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DB_HOST = "localhost"
DB_PORT = 5432
DB_NAME = "Final_Project"
DB_USER = "postgres"
DB_PASSWORD = getpass("PostgreSQL password: ")

print("Project root:", PROJECT_ROOT)
print("Dataset root:", DATASET_ROOT)
print("Output dir :", OUTPUT_DIR)


Project root: C:\Datenanalyse\final Project
Dataset root: C:\Datenanalyse\final Project\Dataset_PlanA-B
Output dir : C:\Datenanalyse\final Project\Output_PlanA-B\09_ML_Bridge_Type_Selection_Explainability_Robustness


## 00A — DATA SOURCE / INPUT–OUTPUT MANIFEST

| Item | Source / origin | Transfer method | Role in Notebook 09 | Destination |
|---|---|---|---|---|
| ML dataset | PostgreSQL `Final_Project` → `final.bridge_ml_dataset_final` | SQL query | Canonical explainability/robustness input | In-memory `df` |
| Geometry | `geom_x` / `geom_y` | SQL + coordinate transform | Latitude/longitude predictors | `work` |
| Traffic | `traffic_dtv_mean` | SQL | DTV predictor | `work` |
| Material | `baustoffklasse` | SQL | Material predictor | `work` |
| Bridge type | `bauwerksart_text` | SQL | Classification target / condition predictor | `work` |
| Condition | `zustandsnote` | SQL | Condition target | `work` |
| Diagnostic outputs | Importance/sensitivity/stability results | Local file write | Audit/review | `Output_PlanA-B/09_ML_Bridge_Type_Selection_Explainability_Robustness` |

### Transfer chain

```text
Notebook 06
    ↓
PostgreSQL: final.bridge_ml_dataset_final
    ↓
Notebook 09
    ├── bridge-level train/test split
    ├── diagnostic classifier/regressor
    ├── permutation importance
    ├── DTV sensitivity
    ├── material sensitivity
    ├── geographic sensitivity
    └── random-seed stability
    ↓
Output_PlanA-B/09_ML_Bridge_Type_Selection_Explainability_Robustness
```

**No BASt/DWD/Traffic download occurs in Notebook 09.**

**No FEM/InfoCAD calculation or structural design is performed.**

**Diagnostic models do not overwrite or retrain the frozen production model.**


## 01 — Load canonical PostgreSQL dataset


In [2]:
## 01 — Load canonical PostgreSQL dataset

SOURCE_TABLE = '"final"."bridge_ml_dataset_final"'

url = URL.create(
    "postgresql+psycopg2",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME,
)
engine = create_engine(url, connect_args={"connect_timeout": 10})

with engine.connect() as conn:
    print("PostgreSQL connection: PASS")
    print("Database:", conn.execute(text("SELECT current_database()")).scalar())

df = pd.read_sql(f"SELECT * FROM {SOURCE_TABLE}", engine)
print("Source table:", SOURCE_TABLE)
print("Shape:", df.shape)
print("Columns:", len(df.columns))


PostgreSQL connection: PASS
Database: Final_Project
Source table: "final"."bridge_ml_dataset_final"
Shape: (52214, 97)
Columns: 97


## 02 — Prepare the analysis table



In [3]:
required = [
    "bridge_id", "geom_x", "geom_y", "traffic_dtv_mean",
    "baustoffklasse", "bauwerksart_text", "zustandsnote",
]
missing = [c for c in required if c not in df.columns]
if missing:
    raise RuntimeError(f"Missing required columns: {missing}")

tr = Transformer.from_crs("EPSG:3857", "EPSG:4326", always_xy=True)
x = pd.to_numeric(df["geom_x"], errors="coerce")
y = pd.to_numeric(df["geom_y"], errors="coerce")
longitude, latitude = tr.transform(x.to_numpy(dtype=float), y.to_numpy(dtype=float))

work = pd.DataFrame({
    "bridge_id": df["bridge_id"].astype(str),
    "latitude": latitude,
    "longitude": longitude,
    "dtv": pd.to_numeric(df["traffic_dtv_mean"], errors="coerce"),
    "bauwerkstoff": df["baustoffklasse"].astype("string").str.strip(),
    "bauwerksart": df["bauwerksart_text"].astype("string").str.strip(),
    "zustandsnote": pd.to_numeric(df["zustandsnote"], errors="coerce"),
}).dropna()

work = work[
    work["zustandsnote"].between(1, 4)
    & work["latitude"].between(-90, 90)
    & work["longitude"].between(-180, 180)
].copy()

print("Usable rows:", f"{len(work):,}")
print("Unique bridge types:", work["bauwerksart"].nunique())
print("Unique materials:", work["bauwerkstoff"].nunique())


Usable rows: 52,214
Unique bridge types: 43
Unique materials: 7


## 03 — Fixed project input contracts



In [4]:
CLS_FEATURES = ["latitude", "longitude", "dtv", "bauwerkstoff"]

REG_FEATURES = [
    "latitude", "longitude", "dtv", "bauwerkstoff", "bauwerksart"
]

print("Classifier inputs:", CLS_FEATURES)
print("Condition-model inputs:", REG_FEATURES)


Classifier inputs: ['latitude', 'longitude', 'dtv', 'bauwerkstoff']
Condition-model inputs: ['latitude', 'longitude', 'dtv', 'bauwerkstoff', 'bauwerksart']


## 04 — Bridge-level train/test split

Explainability is calculated only on an unseen test sample.
The models are therefore not fitted on the observations used for importance measurement.


In [5]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

train_idx, test_idx = next(
    gss.split(work, work["bauwerksart"], groups=work["bridge_id"])
)

train = work.iloc[train_idx].copy()
test = work.iloc[test_idx].copy()

print("Train rows:", f"{len(train):,}")
print("Test rows:", f"{len(test):,}")
print("Train bridges:", train["bridge_id"].nunique())
print("Test bridges:", test["bridge_id"].nunique())
print("Bridge overlap:", len(set(train["bridge_id"]) & set(test["bridge_id"])))

assert len(set(train["bridge_id"]) & set(test["bridge_id"])) == 0


Train rows: 41,771
Test rows: 10,443
Train bridges: 41771
Test bridges: 10443
Bridge overlap: 0


## 05 — Train diagnostic copies

Reduced model size is used only to make explainability practical.
This does not modify the production model architecture defined in the previous notebooks.


In [6]:
cls_pre = ColumnTransformer([
    (
        "num",
        SimpleImputer(strategy="median"),
        ["latitude", "longitude", "dtv"],
    ),
    (
        "cat",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]),
        ["bauwerkstoff"],
    ),
])

classifier = Pipeline([
    ("preprocess", cls_pre),
    ("model", ExtraTreesClassifier(
        n_estimators=200,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        class_weight="balanced",
    )),
])

classifier.fit(
    train[CLS_FEATURES],
    train["bauwerksart"],
)

reg_pre = ColumnTransformer([
    (
        "num",
        SimpleImputer(strategy="median"),
        ["latitude", "longitude", "dtv"],
    ),
    (
        "cat",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]),
        ["bauwerkstoff", "bauwerksart"],
    ),
])

condition_model = Pipeline([
    ("preprocess", reg_pre),
    ("model", ExtraTreesRegressor(
        n_estimators=200,
        min_samples_leaf=5,
        max_features=0.8,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )),
])

condition_model.fit(
    train[REG_FEATURES],
    train["zustandsnote"],
)

print("Diagnostic models: FIT PASS")


Diagnostic models: FIT PASS


## 06 — Fast permutation importance

Only a representative maximum of 2,000 unseen bridges is used and only 2 permutations are performed.

This is a diagnostic speed optimization; it does not alter the ML architecture.


In [7]:
MAX_IMPORTANCE_ROWS = 2000
N_REPEATS = 2

importance_test = test.sample(
    n=min(MAX_IMPORTANCE_ROWS, len(test)),
    random_state=RANDOM_STATE,
)

print("Rows used for permutation importance:", len(importance_test))
print("Permutation repeats:", N_REPEATS)

cls_perm = permutation_importance(
    classifier,
    importance_test[CLS_FEATURES],
    importance_test["bauwerksart"],
    scoring="balanced_accuracy",
    n_repeats=N_REPEATS,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

cls_importance = pd.DataFrame({
    "feature": CLS_FEATURES,
    "importance_mean": cls_perm.importances_mean,
    "importance_std": cls_perm.importances_std,
}).sort_values("importance_mean", ascending=False)

display(cls_importance)


Rows used for permutation importance: 2000
Permutation repeats: 2


,feature,importance_mean,importance_std
0,latitude,0.311946,0.001027
1,longitude,0.290955,0.017040
3,bauwerkstoff,0.240165,0.013245
2,dtv,0.195090,0.007303


In [8]:
reg_perm = permutation_importance(
    condition_model,
    importance_test[REG_FEATURES],
    importance_test["zustandsnote"],
    scoring="neg_mean_absolute_error",
    n_repeats=N_REPEATS,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

reg_importance = pd.DataFrame({
    "feature": REG_FEATURES,
    "importance_mean": reg_perm.importances_mean,
    "importance_std": reg_perm.importances_std,
}).sort_values("importance_mean", ascending=False)

display(reg_importance)


,feature,importance_mean,importance_std
4,bauwerksart,0.055446,0.003790
0,latitude,0.034850,0.001528
1,longitude,0.029395,0.001365
3,bauwerkstoff,0.026848,0.002077
2,dtv,0.015075,0.001041


## 07 — DTV sensitivity


In [9]:
base_project = {
    "latitude": 50.7374,
    "longitude": 7.0982,
    "dtv": 25000.0,
    "bauwerkstoff": "Stahlbeton",
}

common_types = (
    train["bauwerksart"]
    .value_counts()
    .loc[lambda s: s >= 100]
    .index
    .tolist()
)

if not common_types:
    raise RuntimeError("No bridge types meet the minimum support threshold.")

reference_type = common_types[0]

dtv_values = np.array([5000, 10000, 25000, 50000, 75000], dtype=float)

rows = []
for dtv_value in dtv_values:
    inp = pd.DataFrame([{
        **base_project,
        "dtv": dtv_value,
        "bauwerksart": reference_type,
    }])
    pred = float(condition_model.predict(inp[REG_FEATURES])[0])
    rows.append({
        "dtv": dtv_value,
        "bauwerksart": reference_type,
        "predicted_zustandsnote": np.clip(pred, 1, 4),
    })

dtv_sensitivity = pd.DataFrame(rows)
display(dtv_sensitivity)


,dtv,bauwerksart,predicted_zustandsnote
0,5000.0,Plattenbrücke,2.203749
1,10000.0,Plattenbrücke,2.193081
2,25000.0,Plattenbrücke,2.130138
3,50000.0,Plattenbrücke,2.127800
4,75000.0,Plattenbrücke,2.146175


## 08 — Bauwerkstoff sensitivity


In [10]:
materials = train["bauwerkstoff"].value_counts().index.tolist()

rows = []
for material in materials:
    inp = pd.DataFrame([{
        **base_project,
        "bauwerkstoff": material,
        "bauwerksart": reference_type,
    }])
    pred = float(condition_model.predict(inp[REG_FEATURES])[0])
    rows.append({
        "bauwerkstoff": material,
        "bauwerksart": reference_type,
        "predicted_zustandsnote": np.clip(pred, 1, 4),
    })

material_sensitivity = pd.DataFrame(rows)
display(material_sensitivity)


,bauwerkstoff,bauwerksart,predicted_zustandsnote
0,Beton/Stahlbeton,Plattenbrücke,2.132547
1,Spannbeton,Plattenbrücke,2.293148
2,Stahl/Leichtmetall,Plattenbrücke,2.057663
3,Stahlverbund,Plattenbrücke,2.208167
4,Stein,Plattenbrücke,2.131823
5,Holz,Plattenbrücke,2.184254
6,Sonstiger Baustoff,Plattenbrücke,2.130138


## 09 — Geographic sensitivity


In [11]:
unique_locations = train[["latitude", "longitude"]].drop_duplicates()

locations = unique_locations.sample(
    n=min(10, len(unique_locations)),
    random_state=RANDOM_STATE,
).reset_index(drop=True)

rows = []
for _, loc in locations.iterrows():
    inp = pd.DataFrame([{
        "latitude": loc["latitude"],
        "longitude": loc["longitude"],
        "dtv": base_project["dtv"],
        "bauwerkstoff": base_project["bauwerkstoff"],
        "bauwerksart": reference_type,
    }])
    pred = float(condition_model.predict(inp[REG_FEATURES])[0])
    rows.append({
        "latitude": loc["latitude"],
        "longitude": loc["longitude"],
        "predicted_zustandsnote": np.clip(pred, 1, 4),
    })

location_sensitivity = pd.DataFrame(rows)
display(location_sensitivity)


,latitude,longitude,predicted_zustandsnote
0,50.723965,10.836839,2.242287
1,50.848921,9.957111,2.302368
2,48.829155,9.333000,2.225760
3,50.502229,8.682201,2.206092
4,52.228869,11.774384,2.345814
5,54.031665,13.275234,2.270016
6,53.858041,13.685132,2.197320
7,49.209965,7.017329,2.273596
8,48.395340,9.767302,2.235592
9,52.423521,9.593888,2.318625


## 10 — Random-seed stability


In [12]:
seed_results = []

for seed in [1, 7, 21]:
    model_seed = Pipeline([
        ("preprocess", reg_pre),
        ("model", ExtraTreesRegressor(
            n_estimators=200,
            min_samples_leaf=5,
            max_features=0.8,
            random_state=seed,
            n_jobs=-1,
        )),
    ])

    model_seed.fit(train[REG_FEATURES], train["zustandsnote"])

    inp = pd.DataFrame([{
        **base_project,
        "bauwerksart": reference_type,
    }])

    pred = float(model_seed.predict(inp[REG_FEATURES])[0])

    seed_results.append({
        "random_state": seed,
        "predicted_zustandsnote": np.clip(pred, 1, 4),
    })

stability = pd.DataFrame(seed_results)

display(stability)
print(
    "Prediction range:",
    round(stability["predicted_zustandsnote"].min(), 4),
    "to",
    round(stability["predicted_zustandsnote"].max(), 4),
)


,random_state,predicted_zustandsnote
0,1,2.150134
1,7,2.148627
2,21,2.193555


Prediction range: 2.1486 to 2.1936


## 11 — Interpretation

No automatic `best bridge type` label is generated.

The notebook reports model importance, sensitivity and stability for engineering review.
FEM/InfoCAD and structural design remain outside the ML model.


In [13]:
summary = {
    "classifier_features": CLS_FEATURES,
    "condition_model_features": REG_FEATURES,
    "excluded": ["laenge", "breite", "FEM", "InfoCAD"],
    "importance_rows": int(len(importance_test)),
    "importance_repeats": int(N_REPEATS),
    "diagnostic_tree_count": 200,
    "classifier_importance": cls_importance.to_dict(orient="records"),
    "condition_importance": reg_importance.to_dict(orient="records"),
    "stability_prediction_min": float(stability["predicted_zustandsnote"].min()),
    "stability_prediction_max": float(stability["predicted_zustandsnote"].max()),
}

print(json.dumps(summary, indent=2, ensure_ascii=False))


{
  "classifier_features": [
    "latitude",
    "longitude",
    "dtv",
    "bauwerkstoff"
  ],
  "condition_model_features": [
    "latitude",
    "longitude",
    "dtv",
    "bauwerkstoff",
    "bauwerksart"
  ],
  "excluded": [
    "laenge",
    "breite",
    "FEM",
    "InfoCAD"
  ],
  "importance_rows": 2000,
  "importance_repeats": 2,
  "diagnostic_tree_count": 200,
  "classifier_importance": [
    {
      "feature": "latitude",
      "importance_mean": 0.31194583549628485,
      "importance_std": 0.0010267340137823378
    },
    {
      "feature": "longitude",
      "importance_mean": 0.29095501334264945,
      "importance_std": 0.01704039309474431
    },
    {
      "feature": "bauwerkstoff",
      "importance_mean": 0.24016516252286518,
      "importance_std": 0.013245150307597042
    },
    {
      "feature": "dtv",
      "importance_mean": 0.19508976941578032,
      "importance_std": 0.007302903890012674
    }
  ],
  "condition_importance": [
    {
      "feature": "bauwerk

## 12 — Export


In [14]:
## 12 — Export

cls_importance.to_csv(OUTPUT_DIR / "09_classifier_permutation_importance.csv", index=False, encoding="utf-8-sig")
reg_importance.to_csv(OUTPUT_DIR / "09_condition_permutation_importance.csv", index=False, encoding="utf-8-sig")
dtv_sensitivity.to_csv(OUTPUT_DIR / "09_dtv_sensitivity.csv", index=False, encoding="utf-8-sig")
material_sensitivity.to_csv(OUTPUT_DIR / "09_material_sensitivity.csv", index=False, encoding="utf-8-sig")
location_sensitivity.to_csv(OUTPUT_DIR / "09_location_sensitivity.csv", index=False, encoding="utf-8-sig")
stability.to_csv(OUTPUT_DIR / "09_model_stability.csv", index=False, encoding="utf-8-sig")

data_inventory = pd.DataFrame({
    "column_order": range(1, len(work.columns) + 1),
    "column": work.columns,
    "dtype": [str(work[c].dtype) for c in work.columns],
    "missing_count": [int(work[c].isna().sum()) for c in work.columns],
    "role": [
        "group_id" if c == "bridge_id" else
        "classification_target" if c == "bauwerksart" else
        "condition_target" if c == "zustandsnote" else
        "predictor" if c in REG_FEATURES else
        "derived_or_validation"
        for c in work.columns
    ],
})
data_inventory.to_csv(OUTPUT_DIR / "09_analysis_data_inventory.csv", index=False, encoding="utf-8-sig")

manifest = {
    "stage": 9,
    "notebook": "09_ML_Bridge_Type_Selection_Explainability_Robustness",
    "source": SOURCE_TABLE,
    "classifier_features": CLS_FEATURES,
    "condition_model_features": REG_FEATURES,
    "excluded": ["laenge", "breite", "FEM", "InfoCAD"],
    "split": "GroupShuffleSplit by bridge_id",
    "importance_rows": int(len(importance_test)),
    "importance_repeats": int(N_REPEATS),
    "diagnostic_tree_count": 200,
    "purpose": "Explainability and robustness diagnostics only.",
    "production_model_modified": False,
}
(OUTPUT_DIR / "09_data_manifest.json").write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8"
)

human_manifest = (
    "Notebook 09 — ML Bridge-Type Selection — Explainability & Robustness\n"
    "====================================================================\n\n"
    f"PROJECT_ROOT: {PROJECT_ROOT}\n"
    f"SOURCE: PostgreSQL {SOURCE_TABLE}\n"
    f"OUTPUT_DIR: {OUTPUT_DIR}\n\n"
    f"CLASSIFIER_FEATURES: {CLS_FEATURES}\n"
    f"CONDITION_MODEL_FEATURES: {REG_FEATURES}\n"
    "EXCLUDED: laenge, breite, FEM, InfoCAD\n"
    "VALIDATION: GroupShuffleSplit by bridge_id\n"
    "PURPOSE: Explainability and robustness diagnostics only.\n"
    "PRODUCTION_MODEL_MODIFIED: False\n"
)
(OUTPUT_DIR / "09_data_manifest.txt").write_text(human_manifest, encoding="utf-8")

print("09 STATUS: COMPLETE")
print("Output directory:", OUTPUT_DIR)
print("Data manifest:", OUTPUT_DIR / "09_data_manifest.txt")


09 STATUS: COMPLETE
Output directory: C:\Datenanalyse\final Project\Output_PlanA-B\09_ML_Bridge_Type_Selection_Explainability_Robustness
Data manifest: C:\Datenanalyse\final Project\Output_PlanA-B\09_ML_Bridge_Type_Selection_Explainability_Robustness\09_data_manifest.txt
